# Companion Notebook — Chapter 40: Advanced AI Capstone Project
**Intelligent Sensor Monitoring with Machine Learning, Evaluation & Responsible AI**
**Artificial Intelligence Learning Hub — Level 4 (ITERA)**

Notebook end-to-end ini mengeksekusi proyek referensi Capstone di CPU lokal mulai dari perumusan dataset, *zero-leakage split*, pembandingan baseline & kandidat, *explainability*, *error analysis*, kontrol keamanan inferensi, *PSI/KS drift monitoring*, hingga Risk Register dan Model Card.


## Stage 1–2: Dataset Provenance, EDA & Zero-Leakage Splits

Menghasilkan telemetri motor sintetis `1.500` sampel $\times$ `6` fitur (`seed=42`), menyimpan `sensor_data_dictionary.json`, lalu melakukan pemisahan *Train / Validation / Test* berstrata sebelum `StandardScaler` di-*fit* murni pada training set.

In [1]:
import sys
sys.path.append('../code')
import dataset_generator, preprocessing
dataset_generator.main()
preprocessing.main()

CHAPTER 40 CAPSTONE: DATASET & DATA DICTIONARY
Samples: 1500 | Features: 6 | Fault Positive Rate: 26.60%
Saved Data Dictionary to: D:\Workstation\Repo My Corses\Kecerdasan Buatan\content\level-4-advanced-ai\chapter-40\data\sensor_data_dictionary.json

CHAPTER 40 CAPSTONE: ZERO-LEAKAGE DATA PREPARATION
Train Shape : (900, 6) (Fault Rate=26.56%)
Val Shape   : (300, 6) (Fault Rate=26.67%)
Test Shape  : (300, 6) (Fault Rate=26.67%)
Verified: StandardScaler fit strictly on X_train before Val/Test transforms.


## Stage 3–5: Baseline vs Candidate Models & Resource Profiling

Membandingkan `DummyClassifier`, `LogisticRegression`, `RandomForest`, dan `MLPClassifier` pada *Validation Set* berikut profil ukuran artefak (`.pkl`) dan latensi CPU (Mean / P95).

In [2]:
import sys
sys.path.append('../code')
import baseline_model, candidate_models
baseline_model.main()
candidate_models.main()

CHAPTER 40 CAPSTONE: BASELINE MODELS (VALIDATION SET)
- Dummy Majority       | Acc= 73.33% | Prec=0.000 | Rec=0.000 | F1=0.000 | AUC=0.5000
- Logistic Regression  | Acc= 95.33% | Prec=0.893 | Rec=0.938 | F1=0.915 | AUC=0.9897

CHAPTER 40 CAPSTONE: CANDIDATE MODEL BENCHMARK (VALIDATION + CPU RESOURCES)
- LogisticRegression       | Val F1=0.9146 | AUC=0.9897 | Size=  1.15 KB | LatMean=0.0636 ms (P95=0.0766 ms)
- RandomForest(n=50,d=8)   | Val F1=0.8875 | AUC=0.9854 | Size=313.66 KB | LatMean=1.9791 ms (P95=2.6295 ms)
- MLP(32,16)               | Val F1=0.8970 | AUC=0.9883 | Size= 39.22 KB | LatMean=0.0746 ms (P95=0.1253 ms)

Selected Best Model (Validation F1): LogisticRegression
Saved Signed Artifact (SHA-256=2e6ebc8046ef3835...) to: D:\Workstation\Repo My Corses\Kecerdasan Buatan\content\level-4-advanced-ai\chapter-40\experiments\capstone_sensor_model.pkl
D:\Workstation\Repo My Corses\Kecerdasan Buatan\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: Conver

## Stage 6: Final Held-Out Test Evaluation, Explainability & Error Analysis

Memverifikasi SHA-256 artefak pemenang seleksi validasi, lalu mengevaluasi satu kali pada held-out test set (`N=300`) lengkap dengan *Permutation Feature Importance* dan sampel kasus keliru.

In [3]:
import sys
sys.path.append('../code')
import evaluation_and_error_analysis
evaluation_and_error_analysis.main()

CHAPTER 40 CAPSTONE: FINAL HELD-OUT TEST EVALUATION (LogisticRegression)
Test Metrics (N=300 unseen) -> Acc=96.00% | Prec=0.9474 | Rec=0.9000 | F1=0.9231 | ROC-AUC=0.9937
Confusion Matrix (TN, FP / FN, TP) : [[216, 4], [8, 72]] | Total Errors=12
Permutation Importance             : {'temperature_c': 0.0824, 'vibration_rms': 0.1298, 'current_amp': 0.0, 'acoustic_db': 0.0769, 'rpm_deviation': -0.0004, 'voltage_ripple_v': 0.0071}
Sample Misclassified Cases         : [{'test_index': 6, 'true_label': 1, 'pred_label': 0, 'pred_prob': 0.2707, 'temperature_c': 57.79, 'vibration_rms': 2.67}, {'test_index': 87, 'true_label': 1, 'pred_label': 0, 'pred_prob': 0.2417, 'temperature_c': 63.46, 'vibration_rms': 1.99}, {'test_index': 106, 'true_label': 0, 'pred_label': 1, 'pred_prob': 0.6902, 'temperature_c': 59.37, 'vibration_rms': 2.27}]


## Stage 7: Validated Local Inference & Post-Deployment Drift Simulation

Menguji penjagaan skema input (`NaN`/boolean/out-of-bounds) serta menyimulasikan pemantauan aliran telemetri nominal vs tergeser (`+9.5C` & `+1.1 mm/s` getaran) menggunakan **PSI** dan **Kolmogorov-Smirnov**.

In [4]:
import sys
sys.path.append('../code')
import local_inference, monitoring_and_drift
local_inference.main()
monitoring_and_drift.main()

CHAPTER 40 CAPSTONE: VALIDATED LOCAL INFERENCE & FALLBACK
- Nominal Motor            -> {'status': 'OK', 'model_name': 'LogisticRegression', 'predicted_class': 0, 'label': 'NORMAL', 'fault_probability': 0.0022, 'confidence': 0.9978, 'human_escalation': False, 'recommended_action': 'Continue nominal telemetry logging'}
- Overheating & High Vib   -> {'status': 'OK', 'model_name': 'LogisticRegression', 'predicted_class': 1, 'label': 'WARNING_FAULT', 'fault_probability': 1.0, 'confidence': 1.0, 'human_escalation': False, 'recommended_action': 'Manual engineer inspection required'}
- Corrupted NaN Telemetry  -> {'status': 'REJECTED_INVALID_INPUT', 'error': "Non-finite value (NaN/Inf) rejected for 'temperature_c'.", 'human_escalation': True}

CHAPTER 40 CAPSTONE: POST-DEPLOYMENT MONITORING & DRIFT AUDIT
- Nominal Stream                   | Acc= 96.00% | MaxPSI=0.0844 | KS(Temp)=0.0522 | Alert=False
- Drifted Stream (+9.5C/+1.1mm/s)  | Acc= 94.33% | MaxPSI=2.0395 | KS(Temp)=0.4478 | Alert=Tru

## Stage 8: Risk Register, Model Card, Unified Pipeline & Automated Tests

Mencetak **Risk Register** Capstone, mengeksekusi orkestrator terpadu `run_capstone_pipeline.py`, dan memverifikasi suite unit test `test_capstone_suite.py`.

In [5]:
import sys
sys.path.append('../code')
import responsible_ai_assessment, run_capstone_pipeline, unittest, test_capstone_suite
responsible_ai_assessment.main()
run_capstone_pipeline.main()

CHAPTER 40 CAPSTONE: RISK REGISTER & RESPONSIBLE AI SUMMARY
- CAP-R1 [Measure / Manage] Inh=16 -> Res= 4 | Sensor calibration drift causing silent false negatives
- CAP-R2 [Map / Manage    ] Inh=20 -> Res= 3 | Corrupted telemetry payload (NaN/Inf/out-of-bounds)
- CAP-R3 [Govern / Manage ] Inh=10 -> Res= 2 | Pickle deserialization tampering on model artifact
- CAP-R4 [Govern / Map    ] Inh=20 -> Res= 6 | Over-reliance on synthetic training proxy in real machinery

CHAPTER 40 CAPSTONE: END-TO-END INTELLIGENT SENSOR MONITORING PIPELINE
Selected Model    : LogisticRegression (Val-selected; SHA-256=2e6ebc8046ef3835...)
Dummy Baseline    : Val Acc=73.33%, F1=0.0000
Held-Out Test     : Acc=96.00%, F1=0.9231, ROC-AUC=0.9937 (Errors=12/300)
Drift Audit       : Nominal MaxPSI=0.0844 (Alert=False) | Drifted MaxPSI=2.0395 (Alert=True)
Risk & Model Card : 4 Risks Logged | Model Card Saved
Total Runtime     : 1.773s on local CPU
Saved Report JSON : D:\Workstation\Repo My Corses\Kecerdasan Buatan\con